# Семинар 3. Как устроен Transformer: Attention, Encoder/Decoder и KV-cache

В первом семинаре мы превращали текст в числовые признаки (`CountVectorizer`, `TF-IDF`). Во втором научились собирать `Pipeline`, подбирать параметры и познакомились со `spaCy` и Hugging Face. Теперь разберём, **как нейросетевая модель учитывает контекст и почему Transformer способен работать с последовательностью слов**.

> Мы **не обучаем большую языковую модель**: для понимания механизма достаточно маленьких матриц. Блок с предобученной моделью — дополнительный и требует интернета.

**Обновлённая версия:** дополнительные объяснения и мини-эксперименты с опорой на открытые учебные лабораторные. Не требуется читать внешние источники, чтобы выполнить задания в ноутбуке.

## 0. Как работать с ноутбуком

У каждого крупного раздела одинаковая структура: **проблема → теория на словах → простой пример → код → что получилось → задание**.

Вычислительные ячейки запускайте сверху вниз (`Runtime → Run all` в Colab). Большинство работает на CPU без скачивания моделей. Значения в учебных примерах **заданы вручную**: не считайте их реальными весами обученной модели.

**Вопрос до начала:** почему TF-IDF в первом семинаре не различает «Собака укусила человека» и «Человек укусил собаку», если использовать только униграммы?

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

torch.manual_seed(42)
np.set_printoptions(precision=3, suppress=True)
print('PyTorch:', torch.__version__)
print('Устройство:', 'cuda' if torch.cuda.is_available() else 'cpu')

### 📚 Перед началом: увидеть Transformer целиком

**Рекомендую посмотреть (10–15 минут):** [3Blue1Brown — Transformers, the tech behind LLMs](https://www.youtube.com/watch?v=wjZofJX0v4M). Начните с фрагмента **12:27–20:22** про embeddings и представление текста; полный ролик помогает увидеть всю архитектуру. Есть русская аудиодорожка.

**Статья с иллюстрациями:** [The Illustrated Transformer — Jay Alammar](https://jalammar.github.io/illustrated-transformer/) (англ.; на странице есть ссылка на русский перевод). Пока достаточно разделов *A High-Level Look* и *Bringing The Tensors Into The Picture*.

**Перед практикой:** объясните своими словами, почему набор чисел (эмбеддинг) удобнее для нейросети, чем сырая строка.


### Как устроена учебная логика

**Цель:** перейти от уже знакомого TF-IDF к механизму, который учитывает *соседние слова и контекст*. Каждый большой шаг разбираем в одном порядке:

1. **Задача:** какое ограничение возникло?
2. **Интуиция:** пример без формул.
3. **Математика:** вводим только нужные обозначения.
4. **Прогноз:** попробуйте предсказать результат **до запуска** ячейки.
5. **Код и проверка:** сравниваем наблюдение с прогнозом.
6. **Вывод:** что умеет метод и чего он *не* доказывает.

**Понадобится:** Python, NumPy, немного матриц. PyTorch здесь означает просто удобные многомерные массивы. Никаких знаний об обучении нейросетей для первых примеров не требуется.

📓 **Как у других:** [Harvard NLP — The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/) объясняет архитектуру прямо возле реализации; мы используем ту же идею, но начнём с меньших вычислений.

### Знакомая задача из семинара 1: одинаковое слово, разный смысл

Сравните:

- «Он натянул **лук** и выпустил стрелу».
- «Я нарезал **лук** для супа».

В TF-IDF слово «лук» — одна координата словаря. Её значение зависит от частот, но сама координата не становится отдельным представлением значения «оружие» или «овощ».

**Статический embedding** задаёт слову вектор, например `лук → [0.2, −0.5, …]`. Если брать одно и то же словарное представление в обоих предложениях, оно будет одинаковым. **Контекстный embedding** — результат обработки всего предложения: представление токена может отличаться в зависимости от соседних слов.

> Важное различие: Attention — один из механизмов построения контекстных представлений; само по себе наличие вектора ещё не означает, что модель понимает смысл.

**Вопрос до кода:** что именно невозможно выразить одним фиксированным словарным вектором для двух значений слова «лук»?

## 1. От TF-IDF к векторам слов: зачем вообще нужен Transformer?

**Вспоминаем первый семинар.** `TfidfVectorizer` создаёт вектор текста: каждая координата соответствует слову или n-грамме из словаря. Это удобно для классификации, но модель не обязательно понимает значение слова и контекст.

Сравните предложения:

- «Я положил **деньги** в **банк**»;
- «Мы сидели на **берегу** реки у песчаного **банка**».

Здесь одно слово может означать разные вещи. Нам нужно представлять **слово внутри предложения**, учитывая соседние слова.

**Эмбеддинг (embedding)** — обучаемый плотный числовой вектор токена. В отличие от разреженного TF-IDF, его координаты обычно не отвечают отдельным словам словаря. Начальный токен-эмбеддинг не обязан зависеть от контекста: одинаковый токен имеет одинаковый начальный вектор. **Контекстное представление** получается после взаимодействия токенов в слоях модели.

Напомним, что **токен** — не обязательно целое слово: в современных моделях он может быть частью слова или знаком пунктуации. Ниже для простоты токен = слово.

### Маленькая матрица эмбеддингов

Пока не обучаем нейросеть. Представим, что каждому из трёх слов уже соответствует вектор длины 4. Одна строка = один токен, один столбец = одна числовая характеристика.

Форма `X.shape == (3, 4)` означает **3 токена × 4 числа на токен**. Это именно та форма, которую мы передадим механизму внимания.

In [ ]:
tokens = ['Кошка', 'увидела', 'собаку']
X = torch.tensor([[1.0, 0.0, 0.5, 0.0],
                  [0.0, 1.0, 0.5, 0.0],
                  [0.5, 0.0, 1.0, 1.0]])
display(pd.DataFrame(X.numpy(), index=tokens, columns=['x1', 'x2', 'x3', 'x4']))
print('Размер:', tuple(X.shape))

### Шпаргалка по размерам — вместо заучивания букв

| Объект | Что храним | Форма в нашем примере |
|---|---|---|
| `X` | три токена с четырьмя признаками | `(3, 4)` |
| `Q` | какие признаки каждый токен ищет | `(3, 2)` |
| `K` | по каким признакам токены сравниваем | `(3, 2)` |
| `V` | что токен может передать дальше | `(3, 2)` |
| `Q @ K.T` | оценка пары «запрашивающий — источник» | `(3, 3)` |
| `weights @ V` | обновлённый вектор каждого токена | `(3, 2)` |

Запись `(3, 2)` означает **3 строки × 2 числа на токен**, а не «три предложения».

В полном батче появится ещё одно измерение (число текстов). Позже, в Multi-Head Attention, появится измерение голов. Пока намеренно их убираем.

**Проверка понимания.** Если в предложении 10 токенов, а каждый эмбеддинг содержит 128 чисел, какова форма матрицы? Ответ: `(10, 128)`.

### Мини-мост к PyTorch

Мы знаем обычные списки и `numpy`-массивы. `torch.Tensor` — похожая структура данных, но удобная для нейросетей и работы на GPU. Для матричного умножения используем `@`, для транспонирования — `.T`.

В формуле ниже `A @ B.T` означает: сравниваем **каждую строку A с каждой строкой B**.

In [ ]:
A = torch.tensor([[1., 0.], [0., 1.]])
B = torch.tensor([[1., 0.], [1., 1.], [0., 1.]])
print('A:', tuple(A.shape), 'B:', tuple(B.shape))
print('A @ B.T:')
print(A @ B.T)
print('Форма результата:', tuple((A @ B.T).shape))

### Аналогия: вопрос, ярлык, содержимое

Представьте библиотеку. Вы хотите найти информацию о **породе собаки**:

- **Query (Q)** — ваш поисковый запрос: «какая порода?»;
- **Key (K)** — карточка книги, с которой сравнивают запрос;
- **Value (V)** — полезное содержимое, которое извлекают из книги.

В Transformer запросы, ключи и значения — **числовые векторы**. Каждая позиция создаёт свои `Q`, `K`, `V` из входного представления, обычно через обучаемые линейные преобразования:

$$Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V.$$

Матрицы `W` учатся во время обучения; в нашей первой игрушечной демонстрации мы временно используем **срезы `X` вместо обученных проекций**, чтобы сосредоточиться на механике. Не путайте эту учебную подмену с реализацией настоящего Transformer.

**Самопроверка:** почему «что искать» (`Q`) и «какие сведения вернуть» (`V`) — не одно и то же?

[Пример пошаговой лабораторной с Attention](https://colab.research.google.com/github/ugcsdl/ugcsdl.github.io/blob/main/notebooks/Lab_11_Self_Attention_and_Transformer_Block.ipynb) — дополнительное чтение.

**Задание 1.** Поменяйте местами две строки в `B`. Что изменится в столбцах результирующей матрицы? Объясните без запуска кода, затем проверьте.

---

## 2. Механизм внимания: какую информацию брать из других токенов?

Представьте, что мы уточняем представление слова «увидела» в предложении «Кошка увидела собаку». Важны не только данные о слове «увидела», но и то, кто увидел и кого увидели.

**Attention** позволяет каждому токену **взвешенно собрать информацию** от других токенов.

Простой алгоритм:

1. Токен оценивает, какие токены ему важны.
2. Превращает оценки в веса от 0 до 1, суммарно равные 1.
3. Берёт взвешенную сумму векторов информации других токенов.

Ниже это сделаем **без формулы Transformer** — сначала вручную по смыслу.

In [ ]:
importance = torch.tensor([1.0, 2.0, 1.0])
weights_manual = importance / importance.sum()  # 0.25, 0.50, 0.25
weighted_sum = weights_manual @ X
print('Веса:', weights_manual.tolist())
print('Сумма весов:', weights_manual.sum().item())
print('Новый вектор слова «увидела»:', weighted_sum.tolist())

**Что получилось:** мы заменили одно представление взвешенной комбинацией **трёх** представлений. В примере веса заданы руками. В Transformer модель **вычисляет эти веса**, а затем обучает параметры, влияющие на них.

### Откуда появляются Q, K и V?

Представим поиск по библиотеке:

| Термин | Интуиция | Роль |
|---|---|---|
| **Query (Q)** | Что я хочу найти? | Запрос текущего токена |
| **Key (K)** | По каким признакам меня можно найти? | Описание для сравнения |
| **Value (V)** | Какую информацию я передам, если меня выбрали? | Содержимое для взвешенной суммы |

Это **три разные роли**, а не три отдельных слова. Каждый токен получает **все три** вектора. В настоящей модели их строят из входных эмбеддингов обучаемыми матрицами:

$$Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V.$$

`W_Q`, `W_K`, `W_V` — обучаемые параметры. В примере ниже возьмём `Q=K=V=X[:, :2]` только ради прозрачных вычислений; это **не типичная конфигурация обученной модели**.

### 🎬 Видео перед вычислением Q, K и V

**Рекомендуем:** [3Blue1Brown — Attention in transformers, step-by-step](https://www.youtube.com/watch?v=eMlx5fFNoYc&t=89s), особенно **1:39–13:10**. Здесь наглядно объясняется, почему токенам нужно учитывать другие токены и откуда берутся веса внимания. Есть русская аудиодорожка.

**Пошаговая статья:** [The Illustrated Transformer — раздел Self-Attention](https://jalammar.github.io/illustrated-transformer/#self-attention-at-a-high-level). Сначала посмотрите рисунки, затем возвращайтесь к матрицам `Q @ K.T` ниже.

**Вопрос перед кодом:** если `Q` — запрос текущего токена, что именно мы сравниваем с помощью `K`?


### Прежде чем нажать ▶: предсказание

У `Q @ K.T` **строка** соответствует токену, который ищет информацию, а **столбец** — токену, к которому он обращается.

**Сделайте прогноз:** если у нас три токена, сколько получится оценок? Почему именно девять? Что произойдёт с числом оценок, если токенов будет десять?

Код ниже не знает русского языка: числа придуманы для опыта. Поэтому нельзя делать вывод «модель поняла слово кошка» по величине произвольной оценки.

### Шаг 1. Считаем оценки соответствия

`Q @ K.T` — таблица сходств «запрос токена i — ключ токена j». Чем выше число, тем больше *до нормировки* внимание склоняется к токену `j`. В реальной сети значения определяются обученными проекциями и сами по себе не обязаны означать семантическое сходство.

In [ ]:
Q = X[:, :2].clone()
K = X[:, :2].clone()
V = X[:, :2].clone()
scores = Q @ K.T
print('Формы Q, K, V:', tuple(Q.shape), tuple(K.shape), tuple(V.shape))
display(pd.DataFrame(scores.numpy(), index=tokens, columns=tokens).round(2))

### Softmax за 30 секунд: из оценок — в доли

Пусть для одного токена получились оценки `1, 2, 0`. Это **не** доли: сумма равна 3, к тому же в реальности оценки бывают отрицательными.

`softmax` экспоненцирует каждую оценку и делит на сумму экспонент. Чем выше оценка, тем больше итоговый вес; веса положительны и в каждой строке суммируются в 1.

Почему делим оценки на $\sqrt{d_k}$ **до** softmax? При увеличении размерности скалярные произведения могут становиться слишком большими, делая распределение чрезмерно острым и затрудняя обучение.

**Мысленный эксперимент:** при оценках `[2, 2, 2]` какие будут веса? **Ответ:** `[1/3, 1/3, 1/3]`. При `[10, 0, 0]` они будут почти целиком сосредоточены на первой позиции.

In [ ]:
# Дополнение: меняем «резкость» softmax, не обучая модель
example_scores = torch.tensor([2.0, 1.0, 0.0])
for temperature in [0.5, 1.0, 2.0]:
    probs = torch.softmax(example_scores / temperature, dim=-1)
    print(f'Температура {temperature}: {[round(x, 3) for x in probs.tolist()]}')
# Вопрос: при какой температуре веса наиболее равномерны?

### Шаг 2. Softmax: превращаем оценки в вероятностно-подобные веса

Оценки могут быть любыми числами. `softmax` преобразует **каждую строку отдельно** в положительные значения с суммой 1:

$$\operatorname{softmax}(s_i)=\frac{e^{s_i}}{\sum_j e^{s_j}}.$$

Значение веса — **не вероятность того, что токен «правильный»**, а доля внимания в конкретном расчёте. Масштабирование оценок на $\sqrt{d_k}$ предотвращает слишком резкие распределения при больших размерностях ключей.

In [ ]:
d_k = Q.shape[-1]
scaled_scores = scores / math.sqrt(d_k)
attention_weights = torch.softmax(scaled_scores, dim=-1)
display(pd.DataFrame(attention_weights.numpy(), index=tokens, columns=tokens).round(3))
print('Сумма по строкам:', attention_weights.sum(dim=1).tolist())

### Что именно получает токен в результате?

Пусть по отношению к трём источникам веса равны `[0.2, 0.7, 0.1]`. Тогда новое представление — это `0.2 × V_1 + 0.7 × V_2 + 0.1 × V_3`.

Это **взвешенная сумма векторов Value**, а не выбор одного слова и не предсказание следующего токена. Сам Attention преобразует представления; чтобы получить готовую языковую модель, нужны другие компоненты Transformer и обучение.

**Вопрос:** если у одного токена самый большой вес 0.7, означает ли это, что остальные токены полностью отброшены?

### Шаг 3. Получаем контекстные представления

Теперь каждую строку весов умножаем на матрицу **Value**. Выход имеет одну строку на токен и `d_v` чисел в каждой строке:

$$\text{Context}=\text{AttentionWeights}\cdot V.$$

Это ещё не вся Transformer-модель, а **одна операция внимания**. После неё обычно работают выходная проекция, residual connection, нормализация и feed-forward блоки.

In [ ]:
context = attention_weights @ V
print('Форма результата:', tuple(context.shape))
display(pd.DataFrame(context.numpy(), index=tokens, columns=['context_1', 'context_2']).round(3))
fig, ax = plt.subplots(figsize=(5.8, 4))
im = ax.imshow(attention_weights.numpy(), vmin=0, vmax=1, cmap='Blues')
ax.set_xticks(range(3), tokens); ax.set_yticks(range(3), tokens)
ax.set_xlabel('К какому токену обращаемся (Key)'); ax.set_ylabel('Какой токен ищет (Query)')
for i in range(3):
    for j in range(3): ax.text(j, i, f'{attention_weights[i,j]:.2f}', ha='center', va='center')
plt.colorbar(im, ax=ax); plt.title('Веса внимания — учебные данные'); plt.show()

### Теперь вся формула целиком

Мы уже выполнили все её части в коде:

$$\boxed{\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V}$$

- `QKᵀ` — оценки соответствия;
- `√d_k` — масштабирование;
- `M` — **опциональная маска**, которая запрещает определённые связи (вернёмся к ней в Decoder);
- `softmax` — веса, сумма по строке = 1;
- умножение на `V` — сбор информации.

**Следим за формами при T токенах:** `Q:[T,d_k]`, `K:[T,d_k]`, `V:[T,d_v]` → `QKᵀ:[T,T]` → `output:[T,d_v]`.

**Self-Attention** означает, что `Q`, `K`, `V` построены на основе *одной и той же последовательности* (хотя проекции могут быть различными).

### Задание 2. Поэкспериментируем с весами

Поменяйте `Q[1]` (запрос слова «увидела») так, чтобы значение внимания к слову «Кошка» увеличилось. Посчитайте веса снова. Подсказка: изменяйте вектор запроса, а не ключи.

**Контрольный вопрос:** почему мы применили `softmax(..., dim=-1)`, а не ко всей матрице целиком?

In [ ]:
# Место для эксперимента: эта ячейка не меняет Q выше
Q_experiment = Q.clone()
Q_experiment[1] = torch.tensor([3.0, 0.0])
new_weights = torch.softmax((Q_experiment @ K.T) / math.sqrt(d_k), dim=-1)
print('Было:', attention_weights[1].numpy().round(3))
print('Стало:', new_weights[1].numpy().round(3))

### 🎬 Сколько «голов» у внимания?

[3Blue1Brown — Multiple Heads, с отметки 19:19](https://www.youtube.com/watch?v=eMlx5fFNoYc&t=1159s) — рекомендуемый короткий отрывок перед практикой. Проследите, почему разные головы могут выделять разные связи.

**Для углубления:** [The Illustrated Transformer — Multi-Headed Attention](https://jalammar.github.io/illustrated-transformer/#multi-headed-attention).

**Проверьте себя:** две головы — это два разных предложения или два разных способа обработать *одно* предложение?


### Сначала одна голова, затем несколько

Один механизм внимания выдаёт одну матрицу весов. Но для предложения могут быть полезны различные типы зависимостей: согласование, связь действия с объектом, связь местоимения с контекстом.

**Multi-Head Attention** запускает несколько операций внимания с **разными обучаемыми проекциями** `W_Q`, `W_K`, `W_V`, затем объединяет результаты и применяет выходную проекцию.

$$\mathrm{MultiHead}(X)=\mathrm{Concat}(head_1,\ldots,head_h)W_O.$$

Головы *могут* специализироваться на разных закономерностях, но конкретная голова не обязана всегда отвечать за «синтаксис» или «смысл» — это учебная интуиция, не гарантированное свойство.

**Перед кодом:** обе головы получат те же три токена. Почему они всё же способны назначить разные веса?

📓 [Университет Оснабрюка — Multi-Head Attention and the Transformer Block](https://pages.cogsci.uos.de/rrawiel/introdeeplearning/practice/practice09-attention/03-multi-head-attention-and-transformer-block.html): особенно удачное объяснение «один вопрос внимания против нескольких».

---

## 3. Почему одного внимания недостаточно? Multi-Head Attention

**Проблема:** в предложении одновременно могут быть важны разные связи: кто совершает действие, к кому оно относится, как согласуются слова. Один набор весов не обязан удобно представлять все отношения сразу.

**Multi-Head Attention** параллельно запускает несколько механизмов внимания (голов). У каждой головы **свои обучаемые проекции** `W_Q`, `W_K`, `W_V`, поэтому головы *могут* научиться разным связям. Это не гарантируется: головы иногда избыточны.

Пример: `d_model=4`, `num_heads=2`, `head_dim=2`. Каждая голова работает с двумерными Q/K/V, затем результаты соединяются. В настоящем Transformer к объединённому результату дополнительно применяют обучаемую выходную проекцию `W_O`.

**Не путайте:** «многомерное внимание» обычно называют **многоголовым (multi-head)**, а не просто большим числом признаков.

In [ ]:
# Две «головы» для наглядности используют разные пары координат.
# Это учебная иллюстрация: реальные головы используют обучаемые проекции.
heads = []
for left, right in [(0, 2), (2, 4)]:
    x_head = X[:, left:right]
    head_scores = x_head @ x_head.T / math.sqrt(x_head.shape[1])
    head_weights = torch.softmax(head_scores, dim=-1)
    heads.append(head_weights)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
for idx, (ax, w) in enumerate(zip(axes, heads), start=1):
    ax.imshow(w.numpy(), vmin=0, vmax=1, cmap='Blues')
    ax.set_xticks(range(3), tokens, rotation=25); ax.set_yticks(range(3), tokens)
    ax.set_title(f'Голова {idx}')
plt.tight_layout(); plt.show()

**Что заметить:** две головы видят те же три токена, но веса отличаются, потому что сравниваются разные представления.

**Задание 3.** Почему изменение только весов второй головы не обязано менять веса первой? Что происходит с числом векторов после объединения голов?

---

## 4. Где в этой конструкции порядок слов? Positional Encoding

Посмотрите на выражение `Q @ K.T`. Если **одновременно переставить строки входной последовательности**, результат self-attention переставится тем же образом — само по себе оно не понимает, что одно слово было первым, а другое последним. Порядок нужно добавить явно.

В классическом Transformer к эмбеддингам токенов **прибавляют позиционные векторы**. Одна из исторических реализаций — синусоидальное кодирование:

$$PE(pos,2i)=\sin\left(\frac{pos}{10000^{2i/d_{model}}}\right),$$
$$PE(pos,2i+1)=\cos\left(\frac{pos}{10000^{2i/d_{model}}}\right).$$

Вам **не нужно заучивать степени**. Важно понимать: `pos` — позиция токена, `d_model` — размер эмбеддинга, для каждой позиции строится собственный вектор. В современных моделях встречаются и другие методы, например RoPE.

### Мысленный опыт: «собака укусила человека» и «человек укусил собаку»

Если изолированному self-attention дать одни и те же токенные векторы в другом порядке, без информации о позиции механизм не сможет различить смысл порядка так, как требуется в языке.

**Позиционная информация** помогает различать места токенов. В классической версии Transformer к токенным векторам прибавляют синусоидальное positional encoding. В современных моделях встречаются и другие решения, например вращательные позиционные представления (RoPE).

**Важно:** контекстный embedding и positional encoding — разные сущности. Позиция сообщает **где токен**, внимание и последующие слои помогают учесть **какой у него контекст**.

**Вопрос до кода:** если поменять слова местами, должны ли итоговые представления всего предложения остаться прежними?

### 📖 Зачем Transformer нужны позиции?

**Читать с иллюстрациями:** [The Illustrated Transformer — Positional Encoding](https://jalammar.github.io/illustrated-transformer/#representing-the-order-of-the-sequence-using-positional-encoding). Обратите внимание, как информация о позиции добавляется к представлению токена.

**Для углубления:** [Attention Is All You Need](https://arxiv.org/abs/1706.03762) — оригинальная статья о Transformer; раздел 3.5 описывает позиционное кодирование. Для этого семинара читать всю статью необязательно.

**Вопрос перед кодом:** что потеряется, если два предложения содержат одинаковые слова, но в разном порядке, а модель не получает никакой информации о позициях?


In [ ]:
def positional_encoding(length, d_model):
    assert d_model % 2 == 0
    positions = torch.arange(length, dtype=torch.float32).unsqueeze(1)
    div_terms = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe = torch.zeros(length, d_model)
    pe[:, 0::2] = torch.sin(positions * div_terms)
    pe[:, 1::2] = torch.cos(positions * div_terms)
    return pe

pe = positional_encoding(20, 16)
plt.figure(figsize=(9, 3.5)); plt.imshow(pe.numpy(), aspect='auto', cmap='coolwarm')
plt.xlabel('Координата эмбеддинга'); plt.ylabel('Позиция токена'); plt.title('Синусоидальное позиционное кодирование')
plt.colorbar(); plt.show()
print('Позиция 0:', pe[0, :4]); print('Позиция 1:', pe[1, :4])

### Классическая архитектура: сначала не формула, а карта

**Encoder** получает всю входную последовательность и создаёт её контекстные представления. Например, это полезно для анализа текста. В типичном encoder self-attention каждый невырезанный маской токен может учитывать остальные токены входа.

**Decoder** предсказывает выход последовательно и в self-attention **не видит будущие позиции**. В классическом Transformer для перевода Decoder дополнительно использует **cross-attention** к выходу Encoder:

| Механизм | Откуда `Q` | Откуда `K` и `V` |
|---|---|---|
| Encoder self-attention | Encoder | Encoder |
| Decoder causal self-attention | Decoder | Decoder (только разрешённые позиции) |
| Encoder–Decoder cross-attention | Decoder | Encoder |

**Не путайте архитектуры:** BERT — encoder-only, GPT-подобные генеративные модели — decoder-only, исходный Transformer для перевода — encoder–decoder.

**Задание без кода:** что нужно модели для машинного перевода, чего не требует обычная классификация предложения?

**Задание 4.** Как меняются позиционные векторы для позиции 0 и 1? Если два слова имеют одинаковые начальные эмбеддинги, но стоят в разных местах, будут ли одинаковыми суммы `token_embedding + positional_encoding`?

---

## 5. Transformer — не одна операция Attention

Теперь можно собрать полную картину. **Классический Encoder–Decoder Transformer** состоит из слоёв:

**Encoder (понимает входную последовательность):** `Embeddings + Position → Self-Attention → Add & Norm → Feed-Forward → Add & Norm`. Все токены входа обычно могут обращаться ко всем разрешённым токенам.

**Decoder (генерирует выход):** `Embeddings + Position → Masked Self-Attention → Cross-Attention к выходам Encoder → Feed-Forward` с residual connection и нормализацией. В **классическом** encoder–decoder блоке Decoder действительно есть **cross-attention**; в decoder-only моделях вроде GPT отдельного Encoder нет.

**Три основных семейства архитектур:**

| Архитектура | Что делает | Пример |
|---|---|---|
| Encoder-only | Контекстные представления входа; удобно для классификации и извлечения признаков | BERT |
| Decoder-only | Предсказывает следующий токен по предыдущим | GPT-подобные LLM |
| Encoder–Decoder | Преобразует входную последовательность в выходную | T5, классический Transformer |

**Cross-attention** отличается от self-attention: `Q` приходит из Decoder, а `K` и `V` — из Encoder. Таким образом Decoder обращается к информации входа.

### Что такое контекстные эмбеддинги?

Начальный эмбеддинг токена — точка старта. **После слоёв self-attention** он становится функцией *всего доступного контекста*. Поэтому два одинаковых токена «ключ» в «ключ от двери» и «ключ к решению» могут получить разные выходные векторы.

Важное уточнение: это **не гарантия**, что модель идеально различает значения; качество зависит от предобучения и выбранного слоя.

### Causal mask: почему Decoder не должен видеть будущее?

Во время генерации будущие токены ещё не существуют. При обучении Decoder запрещают использовать информацию справа, иначе модель «подсмотрела» бы ответ. **Causal mask** разрешает внимание только к текущему токену и предыдущим.

### 🎬 Decoder и запрет «подглядывать в будущее»

[3Blue1Brown — Masking, с 11:08](https://www.youtube.com/watch?v=eMlx5fFNoYc&t=668s) — короткая визуализация того, как скрываются будущие токены.

**Читать:** [The Illustrated GPT-2 — masked self-attention](https://jalammar.github.io/illustrated-gpt2/). В статье есть последовательные схемы генерации и причинной маски.

**Вопрос:** при генерации третьего токена какие позиции уже доступны модели, а какие ещё не должны быть видны?


In [ ]:
T = 6
allowed = torch.tril(torch.ones(T, T, dtype=torch.bool))
plt.figure(figsize=(5, 4)); plt.imshow(allowed.numpy(), cmap='Greys', vmin=0, vmax=1)
plt.xticks(range(T)); plt.yticks(range(T))
plt.xlabel('Ключ (какой токен можно видеть)'); plt.ylabel('Запрос (кто смотрит)')
plt.title('Causal mask: белое = разрешено'); plt.show()
print(allowed.int())

### Почему causal mask — это не «удаление» слов

При генерации в позиции 3 модель может опираться на токены `0, 1, 2, 3`, но не на позиции `4, 5, …`. Маска меняет **допустимые связи** в матрице внимания, а не стирает слова из исходного набора данных.

Перед softmax запрещённым связям присваивают `−∞`, поэтому их вес равен нулю.

**Предскажите рисунок:** для пяти токенов разрешённые клетки образуют нижний треугольник. У первой строки одна разрешённая клетка; у последней — пять.

После эксперимента объясните, почему применение softmax **до** маски было бы логически неверным.

### Маска в формуле Attention

В запрещённых позициях к оценкам до `softmax` прибавляют `−∞`. Тогда соответствующий вес после `softmax` равен 0. Покажем на случайных учебных векторах.

In [ ]:
q = torch.randn(T, 4); k = torch.randn(T, 4); v = torch.randn(T, 4)
raw = q @ k.T / math.sqrt(4)
masked = raw.masked_fill(~allowed, float('-inf'))
masked_weights = torch.softmax(masked, dim=-1)
assert torch.all(masked_weights[~allowed] == 0)
print('Веса внимания для первого токена:', masked_weights[0].numpy().round(3))
print('Веса внимания для последнего токена:', masked_weights[-1].numpy().round(3))

**Задание 5.** Почему у первого токена при causal mask только одна ненулевая связь? А сколько возможных связей у четвёртого токена (индекс 3)?

### Дополнительно: проверяем контекстные представления на предобученном BERT

Это **необязательный** блок: скачает модель из Hugging Face. Для русского языка используем многоязычный BERT. Один и тот же токен в двух контекстах может иметь разные выходные векторы. Поскольку BERT использует subword-токенизацию, **не будем притворяться, что каждое русское слово — один токен**: посмотрим на выходы всей последовательности.

### 📚 Перед экспериментом с BERT: контекстные эмбеддинги

**Рекомендуем:** [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) — просмотрите блоки про self-attention и прохождение представлений через encoder.

**Для самостоятельного чтения:** [The Illustrated GPT-2](https://jalammar.github.io/illustrated-gpt2/) — сравнение encoder-подхода BERT и decoder-подхода GPT-2. Эксперимент ниже факультативный и загружает модель из интернета.

**Вопрос:** почему в разных предложениях одно и то же слово может получить разные контекстные представления?


In [ ]:
# НЕОБЯЗАТЕЛЬНО: требует интернета и пакета transformers.
# В Colab при необходимости: %pip install -q transformers
from transformers import AutoTokenizer, AutoModel
model_name = 'bert-base-multilingual-cased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).eval()
texts = ['Это ключ от двери.', 'Это ключ к решению задачи.']
encoded = tokenizer(texts, return_tensors='pt', padding=True, truncation=True)
with torch.no_grad():
    result = model(**encoded).last_hidden_state
print('Токены первого предложения:', tokenizer.convert_ids_to_tokens(encoded['input_ids'][0]))
print('Форма [предложений, токенов, признаков]:', tuple(result.shape))
print('Векторы зависят от контекста и слоя модели.')

### 🎬 Как генерация приводит к повторным вычислениям

**Рекомендуем перед разделом:** [3Blue1Brown — Transformers, начало видео](https://www.youtube.com/watch?v=wjZofJX0v4M&t=0s) — схема «предсказать токен → добавить его ко входу → повторить».

**Статья:** [The Illustrated GPT-2 — Language Modeling](https://jalammar.github.io/illustrated-gpt2/) с последовательной иллюстрацией авторегрессионной генерации.

**Вопрос:** если модель уже обработала первые пять токенов, что будет повторно вычислено при наивной обработке первых шести?


### Как рождается ответ LLM: две разные стадии

**Prefill:** модель получает весь начальный запрос (prompt), параллельно обрабатывает его токены с causal mask и создаёт ключи/значения для каждого слоя.

**Decode:** модель генерирует очередной токен; на каждом шаге новый Query сопоставляется с уже накопленными Key/Value. Новые K/V добавляются в кеш.

Без KV-cache реализация могла бы заново прогонять через модель весь уже известный префикс на каждом шаге. **KV-cache позволяет избежать повторных K/V-проекций старых токенов**, но не устраняет необходимость обращать внимание на длинный контекст. И сам кеш растёт по мере генерации.

**Аналогия:** вместо того чтобы каждый раз перечитывать всю книгу для нового вопроса, сохраняем подготовленные карточки старых страниц.

⚠️ KV-cache — приём ускорения **инференса авторегрессионных моделей**. Он не означает, что обучение всех Transformer происходит через тот же цикл генерации.

📓 [Практический ноутбук с prefill и decode](https://colab.research.google.com/github/prashantkul/learn-generative-ai/blob/main/07-autoregressive/notebook.ipynb) — для самостоятельного углубления.

---

## 6. Почему генерация происходит по одному токену?

**Decoder-only модель** получает начальный текст, вычисляет распределение вероятностей следующего токена, выбирает токен, добавляет его к контексту и повторяет процесс.

Например: `Сегодня` → `Сегодня хорошая` → `Сегодня хорошая погода`.

Это называется **авторегрессионной генерацией**. Важная деталь: для следующего токена нужны ключи `K` и значения `V` предыдущих токенов.

**Без кеша:** на каждом шаге мы можем снова пересчитывать `K` и `V` уже обработанного префикса.

**С KV-cache:** храним рассчитанные `K` и `V` предыдущих токенов. На следующем шаге вычисляем `Q, K, V` для нового токена и добавляем новые `K/V` в кеш. Новый `Q` обращается к старым и новым `K/V`.

> KV-cache ускоряет *пошаговый autoregressive inference*, но **занимает дополнительную память**. Он не делает полную attention-матрицу произвольной длины бесплатной и не означает, что вычисления внимания для нового токена исчезают.

### 📖 KV-cache: объяснение от разработчиков Transformers

**Рекомендуем:** [Hugging Face — How caching works](https://huggingface.co/docs/transformers/cache_explanation). Прочитайте начало и раздел *Attention matrices*: там показано, что мы сохраняем предыдущие `K` и `V`, но для нового токена всё равно вычисляем новый `Q` и его внимание ко всему доступному контексту.

**Дополнительно:** [Hugging Face — Cache strategies](https://huggingface.co/docs/transformers/kv_cache) — для тех, кто хочет разобраться в Dynamic/Static Cache и обмене скорости на память.

**Вопрос перед экспериментом:** KV-cache хранит все промежуточные матрицы внимания или только ключи и значения предыдущих токенов?


### Почему проверяем равенство результатов, а не только скорость

Наш эксперимент сравнивает результат для **последнего токена**: (а) пересчитать K/V всех токенов и (б) взять сохранённые K/V предыдущих и вычислить только новые.

При одинаковых входах, весах и causal attention результаты должны совпадать **с точностью численных вычислений**. Если не совпадают — кеш, маска или позиции используются неправильно.

Обратите внимание: это **миниатюрный один слой внимания на случайных данных**, а не полная ускоренная генерация настоящей LLM. Он демонстрирует принцип корректности кеширования, но не даёт честной оценки реального ускорения в секундах.

### Игрушечный эксперимент: пересчитывать весь префикс или сохранить K/V?

Здесь используются случайные **необученные** проекции. Сравниваем **результат внимания последнего токена**. В обоих вариантах вычисляется одна и та же функция; различается объём повторной работы.

In [ ]:
torch.manual_seed(7)
d_model, d_head, length = 16, 8, 12
sequence = torch.randn(length, d_model)
Wq, Wk, Wv = [torch.randn(d_model, d_head) for _ in range(3)]

def last_attention(q_last, all_k, all_v):
    scores = q_last @ all_k.T / math.sqrt(q_last.shape[-1])
    return torch.softmax(scores, dim=-1) @ all_v

# Без кеша: заново строим K/V для всего префикса на каждом шаге.
without_cache = []
for end in range(1, length + 1):
    prefix = sequence[:end]
    without_cache.append(last_attention((prefix @ Wq)[-1:], prefix @ Wk, prefix @ Wv))

# С кешем: каждый K/V считаем ровно один раз.
keys, values, with_cache = [], [], []
for x in sequence:
    current = x.unsqueeze(0)
    keys.append(current @ Wk); values.append(current @ Wv)
    with_cache.append(last_attention(current @ Wq, torch.cat(keys), torch.cat(values)))

print('Выходы совпадают:', all(torch.allclose(a,b,atol=1e-5,rtol=1e-5) for a,b in zip(without_cache, with_cache)))
print('Макс. абсолютная разница:', max((a-b).abs().max().item() for a,b in zip(without_cache, with_cache)))

### Считаем количество обработанных токенов при построении K/V

Это не полный подсчёт операций Transformer, а **только количество токенов, для которых вычисляются проекции K/V**.

- Без кеша при `T` шагах: `1+2+...+T = T(T+1)/2`.
- С кешем: `T`.

Для **реальной модели** выигрыш по времени зависит от реализации, GPU, длины контекста, размера batch и накладных расходов.

In [ ]:
steps = np.arange(1, 101)
no_cache_tokens = steps * (steps + 1) // 2
cached_tokens = steps
plt.figure(figsize=(8, 4))
plt.plot(steps, no_cache_tokens, label='Без кеша: повторно вычисляем K/V')
plt.plot(steps, cached_tokens, label='С кешем: K/V каждого токена один раз')
plt.xlabel('Число сгенерированных шагов'); plt.ylabel('Токенов обработано для K/V (накопленно)')
plt.title('Почему кеш экономит повторные проекции'); plt.legend(); plt.grid(alpha=.25); plt.show()
print(f'100 шагов: без кеша = {no_cache_tokens[-1]}, с кешем = {cached_tokens[-1]}')

**Задание 6.** При 20 токенах сколько проекций K/V мы повторно выполняем без кеша, а сколько с кешем (считайте количество обрабатываемых токенов)? Почему это сравнение не равно точному ускорению по времени?

---

## 7. Почему attention расходует память? KV-cache и FlashAttention

Разделим две **разные** проблемы:

1. **Матрица attention при обработке последовательности**: для `T` токенов полная матрица весов одной головы содержит `T × T` элементов — квадратичный размер `O(T²)`.
2. **KV-cache при пошаговой генерации**: храним ключи и значения для каждого предыдущего токена, обычно на каждом слое. Размер кеша растёт **линейно по длине** контекста `O(T)` при фиксированных параметрах модели.

Оценка памяти кеша (без служебных накладных расходов):

$$\mathrm{bytes}_{KV}=2\cdot B\cdot L\cdot T\cdot H_{kv}\cdot D_{head}\cdot S,$$

где `2` = **K и V**, `B` = batch, `L` = число слоёв, `T` = число закешированных токенов, `H_kv` = число KV-голов, `D_head` = размер головы, `S` = байт на элемент (например, 2 для fp16/bfloat16).

**FlashAttention** решает *другую* задачу: вычисляет attention по блокам, сокращая обмен данными с медленной памятью и избегая материализации полной матрицы весов в HBM. Это не то же самое, что кеширование K/V. Математический attention остаётся тем же (с учётом численной точности).

**Важно:** `O(T²)` выше относится к материализуемой матрице весов; оптимизированная реализация может не хранить её целиком.

### Отдельно считаем цену кеширования

У каждого слоя нужно хранить **два массива**: Keys и Values. Упрощённая оценка объёма:

$$\text{bytes} = 2 \cdot B \cdot L \cdot T \cdot H_{kv} \cdot D \cdot S,$$

где `B` — размер батча, `L` — число слоёв, `T` — число сохранённых токенов, `H_kv` — количество голов для K/V, `D` — размерность одной головы, `S` — байт на элемент. Множитель 2 — **K и V**.

**Важно:** это оценка только самих K/V без накладных расходов, фрагментации и других буферов. При удвоении `T` или `B` объём также удваивается. У GQA может быть меньше KV-голов, чем query-голов, что уменьшает кеш.

**До запуска:** сколько будет памяти для 2048 токенов относительно 512? **Ответ:** в 4 раза больше.

In [ ]:
def kv_cache_mib(batch, layers, tokens, kv_heads, head_dim, bytes_per_element=2):
    total_bytes = 2 * batch * layers * tokens * kv_heads * head_dim * bytes_per_element
    return total_bytes / 1024**2

for t in [512, 2048, 8192]:
    print(f'{t:5d} токенов -> {kv_cache_mib(1, 24, t, 8, 64):8.1f} MiB KV-cache')
print('При batch=4:', round(kv_cache_mib(4, 24, 2048, 8, 64), 1), 'MiB')

**Задание 7.** Увеличьте `batch` вдвое. Как меняется размер KV-cache? Почему использование Grouped-Query Attention (меньше KV-голов, чем Q-голов) позволяет уменьшить кеш?

### Опционально: оптимизированный API PyTorch

`torch.nn.functional.scaled_dot_product_attention` выбирает доступную реализацию внимания в зависимости от устройства и версии PyTorch. Вызывая эту функцию, мы **не гарантируем**, что на CPU запущен именно FlashAttention.

### FlashAttention и KV-cache отвечают на разные вопросы

- **KV-cache:** «Можно ли не пересчитывать уже вычисленные K/V на следующем шаге?» — оптимизация повторных вычислений между шагами генерации.
- **FlashAttention:** «Можно ли считать attention более экономно с точки зрения обмена данными с GPU-памятью?» — использование блочных алгоритмов и избежание материализации всей большой матрицы внимания в основной памяти устройства.

Оба подхода могут использоваться вместе. FlashAttention не означает, что математически исчезли все попарные сравнения токенов.

📓 [Ноутбук IIT Hyderabad по self-attention и Transformer](https://colab.research.google.com/github/DL4CV-NPTEL/2026/blob/main/notebooks/Week%208/W08_L4_Self_Attention_and_Transformers.ipynb) — более продвинутый материал, где реализации сверяют со стандартными модулями PyTorch.

### 📖 Эффективное внимание: чем FlashAttention отличается от KV-cache?

**Документация:** [PyTorch — scaled_dot_product_attention](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html). Это интерфейс к реализации, которая может использовать оптимизированные алгоритмы в зависимости от устройства и входных данных.

**Для углубления (необязательно):** [FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135). Основная идея — уменьшить обращения к памяти устройства, не сохраняя целиком большую матрицу внимания в промежуточной памяти.

**Важно различать:** KV-cache переиспользует рассчитанные `K/V` при *генерации*, а FlashAttention оптимизирует само *вычисление attention*.


In [ ]:
import torch.nn.functional as F
query = torch.randn(1, 2, 5, 8)
key = torch.randn(1, 2, 5, 8)
value = torch.randn(1, 2, 5, 8)
optimized_output = F.scaled_dot_product_attention(query, key, value, is_causal=True)
print('Форма ответа [batch, heads, tokens, head_dim]:', tuple(optimized_output.shape))

---

## 8. Собираем всё вместе: путь одного предложения

**Шаг 1.** Текст → токены → начальные эмбеддинги (с позиционной информацией).

**Шаг 2.** Для каждого токена считаем `Q`, `K`, `V` обучаемыми проекциями.

**Шаг 3.** Attention вычисляет веса и собирает информацию из доступных токенов. Несколько голов делают это параллельно.

**Шаг 4.** Слои Transformer дополнительно преобразуют представления через feed-forward блоки, residual connections и нормализацию. Так появляются контекстные векторы.

**Шаг 5.** Encoder-only модель использует векторы для понимания входа; Decoder генерирует следующий токен. В классической схеме Encoder–Decoder участвуют обе части и cross-attention.

**Шаг 6.** При пошаговой генерации KV-cache повторно использует уже вычисленные ключи/значения. FlashAttention оптимизирует расчёт внимания иначе.

| Механизм | Зачем нужен | Важное ограничение |
|---|---|---|
| TF-IDF | Частотные признаки текста | Не моделирует контекст напрямую |
| Attention | Смешивание информации между токенами | Веса зависят от Q/K; полная матрица T×T |
| Multi-Head | Разные обучаемые проекции | Не гарантирует «разные смыслы» голов |
| Positional Encoding | Информация о порядке | Есть разные реализации |
| Encoder | Представление входного текста | Не предназначен сам по себе для causal генерации |
| Decoder | Генерация последовательности | Требует causal mask для self-attention |
| KV-cache | Не считать прошлые K/V снова | Память растёт с длиной контекста |
| FlashAttention | Снизить memory I/O при attention | Не заменяет KV-cache |

## 9. Самостоятельная работа

Выполняйте **после** всех теоретических блоков, используя уже написанные примеры. Проверяйте не только код, но и объяснение результата.

**Обязательные задания**

1. **Attention.** Для трёх заданных токенов получите матрицу `Q @ K.T`, вычислите `softmax`, проверьте сумму весов по строкам и нарисуйте heatmap. Объясните, что обозначает выбранная ячейка матрицы.
2. **Маска Decoder.** Постройте causal mask для 5 токенов и проверьте, что запрещённые веса после `softmax` равны нулю. Объясните, для чего нужна маска.
3. **KV-cache.** Постройте график стоимости повторных проекций K/V для длины от 1 до 200. Объясните разницу между сохранением K/V и ускоренным вычислением attention.
4. **Рефлексия.** Почему TF-IDF и Transformer решают проблему представления текста по-разному? Чем начальный embedding отличается от контекстного?

**Повышенная сложность**

5. Напишите функцию `my_attention(q, k, v, mask=None)` и проверьте, что её результат совпадает с ручными шагами семинара. Учтите, что `mask` — булева матрица разрешённых позиций.
6. Оцените размер KV-cache для двух моделей с одинаковыми `B,L,T,D_head,S`, но разным числом KV-голов. Какой способ экономии памяти используется?

**Критерии:** корректные формы матриц, корректная нормировка `softmax`, осмысленные выводы и понимание различий между методами.

## 10. Проверяем себя

1. Что уже было известно о тексте из TF-IDF, а что добавляют контекстные представления?
2. В чём роли Query, Key, Value? Откуда берутся обучаемые проекции?
3. Почему `softmax` применяется по каждой строке матрицы внимания?
4. Чем отличаются self-attention и cross-attention?
5. Зачем нужны несколько голов и информация о позиции?
6. Чем отличаются Encoder-only, Decoder-only и Encoder–Decoder?
7. Что запрещает causal mask? Почему она особенно важна при обучении Decoder?
8. Что именно сохраняет KV-cache? Ускоряет ли он каждую операцию?
9. Почему память KV-cache растёт линейно по длине последовательности?
10. Чем FlashAttention отличается от KV-cache?

### Материалы для углубления

- Vaswani et al. (2017), *Attention Is All You Need*: https://arxiv.org/abs/1706.03762
- The Illustrated Transformer (Jay Alammar): https://jalammar.github.io/illustrated-transformer/
- Hugging Face NLP Course: https://huggingface.co/learn/nlp-course/chapter1/1
- PyTorch `scaled_dot_product_attention`: https://pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html
- FlashAttention (Dao et al.): https://arxiv.org/abs/2205.14135

**На следующем занятии** можно перейти к tokenization, обучению/дообучению Transformer или к прикладному retrieval/RAG — фундамент внимания уже разобран.

---
## Материалы после занятия — маршрут повторения

1. **Перед повторением Attention:** [3Blue1Brown — Attention](https://www.youtube.com/watch?v=eMlx5fFNoYc) (26 минут, есть русская аудиодорожка).
2. **Перед повторением Encoder/Decoder:** [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) (иллюстрации, есть ссылка на русский перевод).
3. **Перед повторением KV-cache:** [Hugging Face — How caching works](https://huggingface.co/docs/transformers/cache_explanation).
4. **Для сильных студентов:** [FlashAttention paper](https://arxiv.org/abs/2205.14135) и [PyTorch scaled_dot_product_attention](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html).

Не требуется прочитать всё сразу. Открывайте материалы перед соответствующей практикой; после прочтения попробуйте ответить на вопрос, заданный в блоке.

---
## Дополнительные открытые ноутбуки: что брать после занятия

1. **[The Annotated Transformer — Harvard NLP](https://nlp.seas.harvard.edu/annotated-transformer/)**. Сильная сторона — объяснение архитектуры рядом с реализацией; **уровень: продвинутый**.
2. **[Lab 11: Self-Attention and Transformer Block](https://colab.research.google.com/github/ugcsdl/ugcsdl.github.io/blob/main/notebooks/Lab_11_Self_Attention_and_Transformer_Block.ipynb)**. Хорошо структурированные маленькие шаги по матрицам, маскам и multi-head; **уровень: средний**.
3. **[IIT Hyderabad — Self-Attention and Transformers (2026)](https://colab.research.google.com/github/DL4CV-NPTEL/2026/blob/main/notebooks/Week%208/W08_L4_Self_Attention_and_Transformers.ipynb)**. Проверка самописных частей через PyTorch; **уровень: выше среднего**.
4. **[Multi-Head Attention and Transformer Block — Universität Osnabrück](https://pages.cogsci.uos.de/rrawiel/introdeeplearning/practice/practice09-attention/03-multi-head-attention-and-transformer-block.html)**. Отличная интуиция нескольких голов и residual connections; **уровень: средний**.
5. **[Autoregressive Generation + KV-cache](https://colab.research.google.com/github/prashantkul/learn-generative-ai/blob/main/07-autoregressive/notebook.ipynb)**. Предобработка prompt, поэтапная генерация и кеш; **уровень: продвинутый**.

**Как читать:** сначала выполните наш ноутбук. Внешний материал нужен не вместо объяснения, а чтобы увидеть альтернативную подачу или более полную реализацию. Мы используем педагогические приёмы этих работ, а не копируем их код.